# Olist metadata: configure, create, insert, delete

Defaults: **3 → 5 → 4**. Change the configuration cell to explore other counts and deterministic samples. Observe the same business results in Iceberg, Delta Lake and Hudi. These are the only three operations; no update or schema change precedes delete.

Follow `docs/olist-walkthrough.md` for setup. The CSV is mounted read-only. UTC is a lab parsing convention, not verified source timezone provenance. Deletion is a controlled demonstration.

All host Docker launch commands pin `-p open-table-formats-lab`. CLI stages remain temporary Compose containers; notebook cells use Spark inside this notebook container. Generated outputs stay in ignored local evidence.


## 0. Configuration and a fresh isolated run

Edit only the CONFIG values in the next cell. Initial rows must be positive; insert/delete counts must be nonnegative integers. Deletions cannot exceed initial plus inserted rows, and the source must contain enough distinct records. The seed must be a nonblank string.

The seed ranks source keys independently of CSV order. Initial rows come first; reserved inserts come next. Delete targets start at sample rank 2 and wrap through the selected ranks when needed. Defaults select ranks 1–3 initially, insert ranks 4–5 and delete rank 2. Every run records configuration, source/sample hashes, expected counts and exact target ranks.

Zero inserts or deletes produce an explicitly labeled no-op snapshot, without a table write.


In [ ]:
from pathlib import Path
import sys
project = Path.cwd() if (Path.cwd() / "scripts").exists() else Path.cwd().parent
sys.path.insert(0, str(project / "scripts"))
from olist_lab import OlistLab

# Change these four values to explore the same three-stage experiment.
CONFIG = {
    "initial_rows": 3,
    "insert_rows": 2,
    "delete_rows": 1,
    "sample_seed": "open-table-formats-lab-v1",
}
lab = OlistLab.start(**CONFIG)
lab.source_summary()


## 1. Create the configured initial orders

Defaults create three real source orders independently in each format. Every native business value must match, including null timestamps. Expected counts come from CONFIG and are printed with the actual results below.


In [ ]:
create_result = lab.stage("create")
lab.show(create_result)


### Inspect Iceberg

Follow the current snapshot to its manifest list, then the manifests to data files. Read the decoded Avro records. Snapshot IDs are identifiers, not increasing version numbers. Expand the readable summary, metadata interfaces and actual raw previews. Physical file counts are observed outputs, not assumptions based on row counts.


In [ ]:
lab.show_format(create_result, "iceberg")


### Inspect Delta

Read the latest JSON commit: commitInfo, add/remove actions and file paths. Logical remove actions are distinct from physical file deletion. Expand the readable summary, metadata interfaces and actual raw previews. Physical file counts are observed outputs, not assumptions based on row counts.


In [ ]:
lab.show_format(create_result, "delta")


### Inspect Hudi

Read the completed business-table commit under .hoodie/timeline. Inspect operationType and partitionToWriteStats. Auxiliary files under .hoodie/metadata are counted separately; the business table uses Copy-on-Write. Expand the readable summary, metadata interfaces and actual raw previews. Physical file counts are observed outputs, not assumptions based on row counts.


In [ ]:
lab.show_format(create_result, "hudi")


## 2. Insert the reserved source orders

Defaults insert two distinct reserved orders, giving five total. Other configurations use the specified insert count. Expected difference: only the reserved new orders, with zero updates/deletions. A zero insert count skips the write. No format settings are changed to simplify the result.


In [ ]:
insert_result = lab.stage("insert")
lab.show(insert_result)


### Inspect Iceberg

Follow the current snapshot to its manifest list, then the manifests to data files. Read the decoded Avro records. Snapshot IDs are identifiers, not increasing version numbers. Expand the readable summary, metadata interfaces and actual raw previews. Physical file counts are observed outputs, not assumptions based on row counts.


In [ ]:
lab.show_format(insert_result, "iceberg")


### Inspect Delta

Read the latest JSON commit: commitInfo, add/remove actions and file paths. Logical remove actions are distinct from physical file deletion. Expand the readable summary, metadata interfaces and actual raw previews. Physical file counts are observed outputs, not assumptions based on row counts.


In [ ]:
lab.show_format(insert_result, "delta")


### Inspect Hudi

Read the completed business-table commit under .hoodie/timeline. Inspect operationType and partitionToWriteStats. Auxiliary files under .hoodie/metadata are counted separately; the business table uses Copy-on-Write. Expand the readable summary, metadata interfaces and actual raw previews. Physical file counts are observed outputs, not assumptions based on row counts.


In [ ]:
lab.show_format(insert_result, "hudi")


## 3. Delete the deterministic target orders

Defaults delete sample rank 2, leaving four orders. CONFIG determines other delete counts and target ranks. Expected difference: only those deleted orders, with zero inserts/updates. Zero deletes skip the write. Logical deletion need not remove older physical files immediately.


In [ ]:
delete_result = lab.stage("delete")
lab.show(delete_result)


### Inspect Iceberg

Follow the current snapshot to its manifest list, then the manifests to data files. Read the decoded Avro records. Snapshot IDs are identifiers, not increasing version numbers. Expand the readable summary, metadata interfaces and actual raw previews. Physical file counts are observed outputs, not assumptions based on row counts.


In [ ]:
lab.show_format(delete_result, "iceberg")


### Inspect Delta

Read the latest JSON commit: commitInfo, add/remove actions and file paths. Logical remove actions are distinct from physical file deletion. Expand the readable summary, metadata interfaces and actual raw previews. Physical file counts are observed outputs, not assumptions based on row counts.


In [ ]:
lab.show_format(delete_result, "delta")


### Inspect Hudi

Read the completed business-table commit under .hoodie/timeline. Inspect operationType and partitionToWriteStats. Auxiliary files under .hoodie/metadata are counted separately; the business table uses Copy-on-Write. Expand the readable summary, metadata interfaces and actual raw previews. Physical file counts are observed outputs, not assumptions based on row counts.


In [ ]:
lab.show_format(delete_result, "hudi")


## 4. Compare the three representations

Default progression: **3 → 5 → 4**. Your configured expected progression is recorded in the manifest and printed below. Read business rows, metadata selecting the current state and then file changes. Refresh `http://127.0.0.1:8765` to inspect this saved run by stage and format.

This establishes sequential correctness on the pinned stack, not performance or cross-engine interoperability. Use the manual notebook to inspect updates and the synthetic exercise to explore schema evolution.


In [ ]:
print({"run_id": lab.run_id, "preset": lab.preset_id, "status": lab.manifest["status"],
       "completed_stages": lab.manifest["completed_stages"], "expected_rows": lab.config["counts"],
       "evidence_directory": str(lab.run_dir)})
lab.spark.stop()


## Reset or repeat

Every top-to-bottom execution starts an isolated run; no deletion is needed. The CLI offers a scoped reset only when given the exact run ID. It retains evidence and labels it reset. Earlier runs cannot be written to or reset through this API. Never use the synthetic workflow’s global volume reset for this notebook. After a failed operation, start a fresh run rather than replaying a partial commit.
